# HAM10000 Skin Lesion Classification - ResNet-50 Training

สมุดงานสำหรับฝึกฝนโมเดล ResNet-50 บนชุดข้อมูล HAM10000 (7 คลาส)
- โครงสร้างโมเดล ResNet-50 รวมอยู่ในสมุดงานนี้โดยตรง
- แยกหมวดหมู่พารามิเตอร์: **โครงสร้างโมเดล**, **การฝึกสอน**, และ **ชุดข้อมูล**
- `hidden_dim` ปรับได้ทั้ง **ชั้นเดียว (`None`)**, **2 ชั้น (`512`)**, หรือ **หลายชั้น (`[1024, 512]`)**
- รองรับ **Differential Learning Rates**, **Label Smoothing**, และ **Balanced Class Weights**

In [ ]:
# =========================================================================
# 1. ปรับแต่ง Hyperparameters (แยกหมวดหมู่ชัดเจนสำหรับทดลองจูน)
# =========================================================================

# [หมวด 1: สถาปัตยกรรมและโครงสร้างโมเดล]
MODEL_CONFIG = {
    'model_name': 'resnet50',        # สถาปัตยกรรมโมเดล
    'img_size': 224,                 # ขนาดภาพอินพุต (224 x 224)
    'pretrained': True,              # โหลด ImageNet Pretrained weights
    'dropout_rate': 0.3,             # สัดส่วน Dropout (0.0 ถึง 0.5) ช่วยลด Overfitting
    # ตัวเลือกการปรับ hidden_dim (Dense Layer):
    #   - None              -> ชั้นเดียวตรงๆ (2048 -> 7)
    #   - 512               -> 2 ชั้น (2048 -> 512 -> 7)
    #   - [1024, 512]       -> 3 ชั้น (2048 -> 1024 -> 512 -> 7)
    #   - [1024, 512, 256]  -> 4 ชั้น (2048 -> 1024 -> 512 -> 256 -> 7)
    'hidden_dim': [1024, 512],       # กำหนดขนาด Dense หลายชั้นได้ตามใจชอบ
    'freeze_backbone': False,        # True = แช่แข็ง Conv layers เทรนแค่หัว, False = เทรนทั้งตัว
}

# [หมวด 2: การฝึกสอนและ Optimization เพื่อเพิ่มความแม่นยำ]
TRAIN_CONFIG = {
    'epochs': 15,                    # จำนวนรอบเทรน
    'batch_size': 64,                # ขนาด Batch (64 เหมาะกับ VRAM 24GB)
    'head_lr': 1e-4,                 # Learning Rate สำหรับส่วนหัว (Dense Layer)
    'backbone_lr': 1e-5,             # Learning Rate สำหรับ Backbone (ต่ำกว่า 10 เท่าเพื่อคง Feature เดิม)
    'weight_decay': 1e-4,            # L2 Regularization
    'label_smoothing': 0.1,          # ป้องกันโมเดลมั่นใจผิดจุดในภาพยาก
    'use_amp': True,                 # เปิดใช้ Automatic Mixed Precision (bf16)
}

# [หมวด 3: ชุดข้อมูล การชดเชยคลาส และระบบเครื่อง]
DATA_CONFIG = {
    'use_class_weights': True,       # คำนวณ Balanced Class Weights ใน Loss Function
    'use_weighted_sampler': False,   # สุ่ม batch ตามสัดส่วนผกผัน (เปิดใช้เมื่อไม่ใช้ class weights)
    'cache_in_ram': False,           # แคชภาพ uint8 ลง RAM (ช่วยรอบ Epoch ไวขึ้นมาก)
    'num_workers': 4,                # จำนวน CPU threads ดึงภาพ
    'seed': 42,                      # Random Seed ผลลัพธ์ซ้ำได้
    'checkpoint_dir': 'checkpoints', # โฟลเดอร์บันทึกโมเดลที่ดีที่สุด
    'splits_file': 'splits/splits.csv',
    'data_dir': 'data',
}

print('Configurations loaded successfully!')

In [ ]:
# =========================================================================
# 2. ตรวจสอบสเปก GPU และล็อก Seed
# =========================================================================
from pathlib import Path
from typing import Optional, Union, List
import torch
import torch.nn as nn
import torchvision.models as models
from torch.utils.data import DataLoader
from sklearn.metrics import classification_report, f1_score

from src.utils import print_env_info, set_seed, get_amp_context
from src.augment import get_train_transform, get_val_transform
from src.dataset import HAM10000Dataset, CLASSES, get_class_weights, get_weighted_sampler

set_seed(DATA_CONFIG['seed'])
print_env_info()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
Path(DATA_CONFIG['checkpoint_dir']).mkdir(parents=True, exist_ok=True)

In [ ]:
# =========================================================================
# 3. สร้าง Dataset และ DataLoader (Train / Val)
# =========================================================================
train_transform = get_train_transform(img_size=MODEL_CONFIG['img_size'])
val_transform = get_val_transform(img_size=MODEL_CONFIG['img_size'])

train_ds = HAM10000Dataset(
    splits_csv=DATA_CONFIG['splits_file'],
    split='train',
    data_dir=DATA_CONFIG['data_dir'],
    transform=train_transform,
    cache_in_ram=DATA_CONFIG['cache_in_ram'],
    img_size=MODEL_CONFIG['img_size'],
)

val_ds = HAM10000Dataset(
    splits_csv=DATA_CONFIG['splits_file'],
    split='val',
    data_dir=DATA_CONFIG['data_dir'],
    transform=val_transform,
    cache_in_ram=DATA_CONFIG['cache_in_ram'],
    img_size=MODEL_CONFIG['img_size'],
)

sampler = get_weighted_sampler(train_ds, split='train') if DATA_CONFIG['use_weighted_sampler'] else None
shuffle = (sampler is None)

train_loader = DataLoader(
    train_ds,
    batch_size=TRAIN_CONFIG['batch_size'],
    shuffle=shuffle,
    sampler=sampler,
    num_workers=DATA_CONFIG['num_workers'],
    pin_memory=True,
    persistent_workers=(DATA_CONFIG['num_workers'] > 0),
)

val_loader = DataLoader(
    val_ds,
    batch_size=TRAIN_CONFIG['batch_size'],
    shuffle=False,
    num_workers=DATA_CONFIG['num_workers'],
    pin_memory=True,
    persistent_workers=(DATA_CONFIG['num_workers'] > 0),
)

print(f"Train samples: {len(train_ds)} | Val samples: {len(val_ds)}")

In [ ]:
# =========================================================================
# 4. โครงสร้างโมเดล ResNet-50 ในตัว (รองรับ Dense หลายชั้น)
# =========================================================================
def build_resnet50(
    num_classes: int = 7,
    pretrained: bool = True,
    dropout_rate: float = 0.3,
    hidden_dim: Optional[Union[int, List[int]]] = None,
    freeze_backbone: bool = False,
) -> nn.Module:
    weights = models.ResNet50_Weights.DEFAULT if pretrained else None
    model = models.resnet50(weights=weights)

    if freeze_backbone:
        for param in model.parameters():
            param.requires_grad = False

    in_features = model.fc.in_features

    # รองรับทั้ง None, int เดี่ยว, หรือ List หลายชั้น (เช่น [1024, 512])
    if hidden_dim is not None:
        dims = [hidden_dim] if isinstance(hidden_dim, int) else list(hidden_dim)
        layers = []
        prev_dim = in_features
        for d in dims:
            layers.extend([
                nn.Linear(prev_dim, d),
                nn.BatchNorm1d(d),
                nn.ReLU(inplace=True),
                nn.Dropout(p=dropout_rate),
            ])
            prev_dim = d
        layers.append(nn.Linear(prev_dim, num_classes))
        model.fc = nn.Sequential(*layers)
    else:
        if dropout_rate > 0.0:
            model.fc = nn.Sequential(
                nn.Dropout(p=dropout_rate),
                nn.Linear(in_features, num_classes),
            )
        else:
            model.fc = nn.Linear(in_features, num_classes)

    return model

# สร้างโมเดลตาม Config
model = build_resnet50(
    num_classes=len(CLASSES),
    pretrained=MODEL_CONFIG['pretrained'],
    dropout_rate=MODEL_CONFIG['dropout_rate'],
    hidden_dim=MODEL_CONFIG['hidden_dim'],
    freeze_backbone=MODEL_CONFIG['freeze_backbone'],
).to(device)

print("Model Head (model.fc):\n", model.fc)

# Loss Function พร้อม Class Weights และ Label Smoothing
loss_weight = get_class_weights(DATA_CONFIG['splits_file'], split='train').to(device) if DATA_CONFIG['use_class_weights'] else None
criterion = nn.CrossEntropyLoss(weight=loss_weight, label_smoothing=TRAIN_CONFIG['label_smoothing'])

# Differential Learning Rates: Backbone LR vs Head LR
if MODEL_CONFIG['freeze_backbone']:
    optimizer = torch.optim.AdamW(model.fc.parameters(), lr=TRAIN_CONFIG['head_lr'], weight_decay=TRAIN_CONFIG['weight_decay'])
else:
    backbone_params = [p for n, p in model.named_parameters() if not n.startswith('fc') and p.requires_grad]
    head_params = [p for n, p in model.named_parameters() if n.startswith('fc') and p.requires_grad]
    optimizer = torch.optim.AdamW([
        {'params': backbone_params, 'lr': TRAIN_CONFIG['backbone_lr']},
        {'params': head_params, 'lr': TRAIN_CONFIG['head_lr']},
    ], weight_decay=TRAIN_CONFIG['weight_decay'])

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=TRAIN_CONFIG['epochs'])
print('Optimizer, Scheduler & Loss ready!')

In [ ]:
# =========================================================================
# 5. Training & Validation Loop
# =========================================================================
best_val_f1 = 0.0
model_name = MODEL_CONFIG['model_name']
best_ckpt_path = Path(DATA_CONFIG['checkpoint_dir']) / f"{model_name}_best.pt"

for epoch in range(1, TRAIN_CONFIG['epochs'] + 1):
    # --- Train ---
    model.train()
    train_loss, train_correct, total_train = 0.0, 0, 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device, non_blocking=True), labels.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with get_amp_context(use_amp=TRAIN_CONFIG['use_amp']):
            outputs = model(imgs)
            loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * len(labels)
        train_correct += (outputs.argmax(dim=1) == labels).sum().item()
        total_train += len(labels)
    scheduler.step()

    # --- Validation ---
    model.eval()
    val_loss, val_preds, val_targets = 0.0, [], []
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device, non_blocking=True), labels.to(device, non_blocking=True)
            with get_amp_context(use_amp=TRAIN_CONFIG['use_amp']):
                outputs = model(imgs)
                loss = criterion(outputs, labels)
            val_loss += loss.item() * len(labels)
            val_preds.extend(outputs.argmax(dim=1).cpu().tolist())
            val_targets.extend(labels.cpu().tolist())

    train_acc = train_correct / total_train
    val_acc = (torch.tensor(val_preds) == torch.tensor(val_targets)).float().mean().item()
    val_f1 = f1_score(val_targets, val_preds, average='macro')

    total_epochs = TRAIN_CONFIG['epochs']
    print(f"Epoch {epoch:02d}/{total_epochs:02d} | Train Loss: {train_loss/total_train:.4f} Acc: {train_acc:.4f} | Val Loss: {val_loss/len(val_ds):.4f} Acc: {val_acc:.4f} Macro-F1: {val_f1:.4f}")

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save({
            'epoch': epoch,
            'model_state': model.state_dict(),
            'optimizer_state': optimizer.state_dict(),
            'val_f1': val_f1,
            'model_config': MODEL_CONFIG,
            'train_config': TRAIN_CONFIG,
        }, best_ckpt_path)
        print(f"  >> Saved new best checkpoint to {best_ckpt_path} (Macro-F1: {best_val_f1:.4f})")

print('Training complete! Best Val Macro-F1:', round(best_val_f1, 4))

In [ ]:
# =========================================================================
# 6. ประเมินผลบน Test Set (Held-out Test)
# =========================================================================
print(f"Loading best checkpoint: {best_ckpt_path}")
ckpt = torch.load(best_ckpt_path, map_location=device)
model.load_state_dict(ckpt['model_state'])
model.eval()

test_ds = HAM10000Dataset(
    splits_csv=DATA_CONFIG['splits_file'],
    split='test',
    data_dir=DATA_CONFIG['data_dir'],
    transform=val_transform,
)
test_loader = DataLoader(test_ds, batch_size=TRAIN_CONFIG['batch_size'], shuffle=False, num_workers=DATA_CONFIG['num_workers'])

test_preds, test_targets = [], []
with torch.no_grad():
    for imgs, labels in test_loader:
        imgs = imgs.to(device, non_blocking=True)
        with get_amp_context(use_amp=TRAIN_CONFIG['use_amp']):
            outputs = model(imgs)
        test_preds.extend(outputs.argmax(dim=1).cpu().tolist())
        test_targets.extend(labels.tolist())

print('\nClassification Report on TEST SET:')
print(classification_report(test_targets, test_preds, target_names=CLASSES, digits=4))